# 🔗 Production-Ready RAG Pipeline with Deduplication & Versioning

This notebook builds a complete **Retrieval-Augmented Generation (RAG)** pipeline from local files.
It handles real-world data engineering problems:
- ✅ Duplicate file detection via SHA256 hashing
- ✅ Chunk-level deduplication
- ✅ File versioning & re-ingestion safety
- ✅ Metadata tracking per chunk
- ✅ ChromaDB via Docker REST API
- ✅ Sentence-Transformers embeddings (`all-MiniLM-L6-v2`)
- ✅ RAG retrieval + prompt builder

**Prerequisites:** ChromaDB running on `localhost:8000` via Docker.

---
## Cell 1 — Install Required Libraries

In [ ]:
# Install all required dependencies
# Run this cell once; restart the kernel if any package is newly installed.

%pip install --quiet \
    chromadb \
    sentence-transformers \
    langchain \
    langchain-community \
    langchain-text-splitters \
    pymupdf \
    python-docx \
    tqdm \
    numpy

---
## Cell 2 — Imports & Global Configuration

In [16]:
# -- Standard library ---------------------------------------------------------
import os
import re
import json
import math
import hashlib
import logging
import unicodedata
from pathlib import Path
from datetime import datetime, timezone
from typing import Optional
from collections import Counter, defaultdict

# -- Third-party --------------------------------------------------------------
import numpy as np
from tqdm import tqdm

# -- Document loaders ---------------------------------------------------------
try:
    import fitz  # PyMuPDF: robust PDF text extraction
except ImportError as e:
    raise ImportError("Missing PyMuPDF. Run the install cell first: %pip install pymupdf") from e

try:
    import pypdf  # Optional fallback for PDFs that PyMuPDF cannot open
except ImportError:
    pypdf = None

try:
    from docx import Document as DocxDoc  # DOCX parsing
except ImportError as e:
    raise ImportError("Missing python-docx. Run the install cell first: %pip install python-docx") from e

# -- LangChain ----------------------------------------------------------------
try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ImportError as e:
    raise ImportError(
        "Missing langchain-text-splitters. Run the install cell first: "
        "%pip install langchain-text-splitters"
    ) from e

# -- Embeddings ---------------------------------------------------------------
try:
    from sentence_transformers import SentenceTransformer
except ImportError as e:
    raise ImportError(
        "Missing sentence-transformers. Run the install cell first: "
        "%pip install sentence-transformers"
    ) from e

# -- ChromaDB (HTTP client, connects to Docker) -------------------------------
try:
    import chromadb
except ImportError as e:
    raise ImportError("Missing chromadb. Run the install cell first: %pip install chromadb") from e

# -- Logging ------------------------------------------------------------------
logger = logging.getLogger("rag_ingestion")
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("%(levelname)s | %(message)s"))
    logger.addHandler(handler)
logger.setLevel(logging.INFO)
logger.propagate = False

# -----------------------------------------------------------------------------
# GLOBAL CONFIGURATION  (edit these to match your environment)
# -----------------------------------------------------------------------------
CONFIG = {
    "data_dir": "../data-RAG",
    "chroma_host": "localhost",
    "chroma_port": 8000,
    "collection_name": "rag_collection_app",
    "chunk_size": 800,
    "chunk_overlap": 100,
    "min_page_chars": 40,
    "min_page_words": 3,
    "min_chunk_chars": 120,
    "min_chunk_words": 8,
    "max_numeric_ratio": 0.55,
    "max_private_use_ratio": 0.02,
    "header_footer_min_page_ratio": 0.60,
    "header_footer_window": 3,
    "embedding_batch_size": 64,
    "chroma_batch_size": 500,
    "existing_hash_batch_size": 5000,
    "embedding_model": "all-MiniLM-L6-v2",
    "registry_path": "./ingestion_registry.json",
    "near_dup_threshold": 0.97,
    "near_dup_probe_chars": 1500,
    "top_k": 5,
}

print("Configuration loaded:")
for k, v in CONFIG.items():
    print(f"   {k}: {v}")

Configuration loaded:
   data_dir: ../data-RAG
   chroma_host: localhost
   chroma_port: 8000
   collection_name: rag_collection_app
   chunk_size: 800
   chunk_overlap: 100
   min_page_chars: 40
   min_page_words: 3
   min_chunk_chars: 120
   min_chunk_words: 8
   max_numeric_ratio: 0.55
   max_private_use_ratio: 0.02
   header_footer_min_page_ratio: 0.6
   header_footer_window: 3
   embedding_batch_size: 64
   chroma_batch_size: 500
   existing_hash_batch_size: 5000
   embedding_model: all-MiniLM-L6-v2
   registry_path: ./ingestion_registry.json
   near_dup_threshold: 0.97
   near_dup_probe_chars: 1500
   top_k: 5


---
## Cell 3 — Registry: Load / Save Ingestion State

The **ingestion registry** is a local JSON file that maps each file hash to its
metadata. This is the single source of truth for deduplication and versioning.
It survives process restarts and ChromaDB wipes.

In [17]:
# -----------------------------------------------------------------------------
# Ingestion Registry Helpers
# Structure: { file_hash: { filename, path, version, ingested_at, chunk_ids } }
# -----------------------------------------------------------------------------

def load_registry(path: str) -> dict:
    """Load the JSON registry from disk; return empty dict if not found."""
    registry_path = Path(path)
    if not registry_path.exists():
        return {}

    try:
        with open(registry_path, "r", encoding="utf-8") as f:
            return json.load(f)
    except json.JSONDecodeError as e:
        raise ValueError(f"Registry file is not valid JSON: {registry_path}") from e


def save_registry(registry: dict, path: str) -> None:
    """Persist the registry with a temp-file replace to avoid partial writes."""
    registry_path = Path(path)
    registry_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = registry_path.with_suffix(registry_path.suffix + ".tmp")

    with open(tmp_path, "w", encoding="utf-8") as f:
        json.dump(registry, f, indent=2, ensure_ascii=False)
        f.write("\n")

    os.replace(tmp_path, registry_path)


def get_file_version(registry: dict, filename: str) -> int:
    """
    Return the next version number for a given filename.
    Scans all registry entries for this filename (different hashes = new version).
    """
    versions = [
        int(entry.get("version", 0))
        for entry in registry.values()
        if entry.get("filename") == filename
    ]
    return max(versions) + 1 if versions else 1


REGISTRY = load_registry(CONFIG["registry_path"])
print(f"Registry loaded. Tracked files: {len(REGISTRY)}")

Registry loaded. Tracked files: 23


---
## Cell 4 — Hashing Utilities

In [18]:
# -----------------------------------------------------------------------------
# Hashing Utilities
#   - sha256_file: raw file hash for exact file-level deduplication
#   - sha256_string: text hash for chunk-level deduplication
# -----------------------------------------------------------------------------

def sha256_file(filepath: str) -> str:
    """
    Compute SHA-256 hash of a file from raw bytes.
    Reading in 64 KB blocks keeps memory usage constant for large files.
    """
    h = hashlib.sha256()
    with open(filepath, "rb") as f:
        for block in iter(lambda: f.read(65536), b""):
            h.update(block)
    return h.hexdigest()


def normalize_for_hash(text: str) -> str:
    """Canonicalize text so whitespace/case-only changes do not duplicate vectors."""
    text = unicodedata.normalize("NFKC", text or "")
    return re.sub(r"\s+", " ", text).strip().casefold()


def sha256_string(text: str, *, canonicalize: bool = False) -> str:
    """Compute SHA-256 hash of a UTF-8 string."""
    value = normalize_for_hash(text) if canonicalize else (text or "")
    return hashlib.sha256(value.encode("utf-8")).hexdigest()


def is_file_already_ingested(file_hash: str, registry: dict) -> bool:
    """Return True if this exact file hash exists in the registry."""
    return file_hash in registry


test_hash = sha256_string("hello world")
print(f"SHA-256 of 'hello world': {test_hash[:32]}...")

SHA-256 of 'hello world': b94d27b9934d3e08a52e52d7da7dabfa...


---
## Cell 5 — Document Loaders (PDF / TXT / DOCX)

In [19]:
# -----------------------------------------------------------------------------
# Document Loaders
#   Each loader returns a list of dicts: { "text": str, "page": int|None }
#   Page numbers let us reconstruct provenance in metadata.
# -----------------------------------------------------------------------------

def _page_get_text(page, mode: str, **kwargs):
    """Call PyMuPDF get_text with compatibility for older versions."""
    try:
        return page.get_text(mode, **kwargs) or ""
    except TypeError:
        kwargs.pop("sort", None)
        return page.get_text(mode, **kwargs) or ""


def _extract_pdf_page_text_pymupdf(page) -> tuple[str, str]:
    """
    Extract text from one PDF page with PyMuPDF.
    Sorted text blocks usually preserve reading order better than a plain dump.
    """
    blocks = _page_get_text(page, "blocks", sort=True)
    pieces: list[str] = []

    if blocks:
        for block in sorted(blocks, key=lambda b: (round(float(b[1]), 1), round(float(b[0]), 1))):
            if len(block) >= 5 and isinstance(block[4], str) and block[4].strip():
                pieces.append(block[4].strip())

    if pieces:
        return "\n".join(pieces), "pymupdf_blocks"

    text = _page_get_text(page, "text", sort=True).strip()
    return text, "pymupdf_text"


def _load_pdf_with_pypdf(filepath: str) -> list[dict]:
    """Fallback PDF extraction with pypdf, if installed."""
    if pypdf is None:
        return []

    pages = []
    try:
        reader = pypdf.PdfReader(filepath)
        for page_num, page in enumerate(reader.pages, start=1):
            text = page.extract_text() or ""
            if text.strip():
                pages.append({"text": text, "page": page_num, "extraction_method": "pypdf_fallback"})
    except Exception as e:
        logger.warning("pypdf fallback failed for '%s': %s", Path(filepath).name, e)
    return pages


def load_pdf(filepath: str) -> list[dict]:
    """Extract text page-by-page from a PDF using PyMuPDF, with pypdf fallback."""
    pages = []
    blank_pages = 0

    try:
        with fitz.open(filepath) as doc:
            for page_num, page in enumerate(doc, start=1):
                text, method = _extract_pdf_page_text_pymupdf(page)
                if text.strip():
                    pages.append({"text": text, "page": page_num, "extraction_method": method})
                else:
                    blank_pages += 1
    except Exception as e:
        logger.warning("PyMuPDF could not read PDF '%s': %s", Path(filepath).name, e)

    if not pages:
        fallback_pages = _load_pdf_with_pypdf(filepath)
        if fallback_pages:
            pages = fallback_pages

    if blank_pages:
        logger.info("PDF extraction: %s had %d blank/image-only page(s)", Path(filepath).name, blank_pages)

    if not pages:
        logger.warning(
            "PDF extraction produced no text for '%s'. It may be scanned, image-only, or require OCR before ingestion.",
            Path(filepath).name,
        )

    return pages


def load_txt(filepath: str) -> list[dict]:
    """Read a plain-text file as a single document block."""
    for encoding in ("utf-8", "utf-8-sig", "cp1252", "latin-1"):
        try:
            with open(filepath, "r", encoding=encoding) as f:
                text = f.read()
            if text.strip():
                return [{"text": text, "page": None, "extraction_method": f"txt:{encoding}"}]
            return []
        except UnicodeDecodeError:
            continue
        except Exception as e:
            logger.warning("Could not read TXT '%s': %s", Path(filepath).name, e)
            return []

    logger.warning("Could not decode TXT '%s' with known encodings", Path(filepath).name)
    return []


def load_docx(filepath: str) -> list[dict]:
    """Extract paragraph and table text from a DOCX file."""
    try:
        doc = DocxDoc(filepath)
        blocks: list[str] = []

        for para in doc.paragraphs:
            if para.text.strip():
                blocks.append(para.text.strip())

        for table in doc.tables:
            for row in table.rows:
                cells = [cell.text.strip() for cell in row.cells if cell.text.strip()]
                if cells:
                    blocks.append(" | ".join(cells))

        full_text = "\n".join(blocks)
        if full_text.strip():
            return [{"text": full_text, "page": None, "extraction_method": "python-docx"}]
    except Exception as e:
        logger.warning("Could not read DOCX '%s': %s", Path(filepath).name, e)
    return []


def load_document(filepath: str) -> list[dict]:
    """Route a file to the appropriate loader based on its extension."""
    ext = Path(filepath).suffix.lower()
    if ext == ".pdf":
        return load_pdf(filepath)
    if ext == ".txt":
        return load_txt(filepath)
    if ext == ".docx":
        return load_docx(filepath)

    logger.warning("Unsupported file type %s; skipping '%s'", ext, filepath)
    return []


def discover_files(data_dir: str) -> list[str]:
    """Recursively find all PDF, TXT, DOCX files under data_dir."""
    supported_ext = {".pdf", ".txt", ".docx"}
    data_path = Path(data_dir)
    if not data_path.exists():
        logger.warning("Data directory does not exist: %s", data_path.resolve())
        return []

    files = [str(p) for p in data_path.rglob("*") if p.is_file() and p.suffix.lower() in supported_ext]
    return sorted(files)


all_files = discover_files(CONFIG["data_dir"])
print(f"Discovered {len(all_files)} supported file(s):")
for f in all_files:
    print(f"   {f}")

Discovered 25 supported file(s):
   ..\data-RAG\130042010150605.pdf
   ..\data-RAG\130828-veille_huile_dolive-sl.pdf
   ..\data-RAG\19-00145-book_agricultures_en_chiffres_def.pdf
   ..\data-RAG\19-00145-siam-book_formation18x22vavf_9.pdf
   ..\data-RAG\20-00529-MA_Plaquette_Bilan PMV_VF(6-7-21)-compressé_0.pdf
   ..\data-RAG\ATLASsynthese.pdf
   ..\data-RAG\Agriculture familiale - Fiche stratégique VF déf.pdf
   ..\data-RAG\Book SIAM - La Protection Sociale des Agriculteurs VF.pdf
   ..\data-RAG\ESSA Transforming Agrifood 21 Nov 2024 final-V3.docx
   ..\data-RAG\Filière Truffes au Maroc VF btt_213.pdf
   ..\data-RAG\Loi 58-12 - Création ONCA - Langue française.pdf
   ..\data-RAG\Mécanisation agricole VF_btt_209.pdf
   ..\data-RAG\PGES-Accord de financement-PAASIFEJ -BAD vf.pdf
   ..\data-RAG\Rapport SNIF 2022 V 03042024.pdf
   ..\data-RAG\Valorisation eau VF_btt_211.pdf
   ..\data-RAG\agriculture-en-chiffres-2012.pdf
   ..\data-RAG\agriculture_maroc_fruits_synthetic-data.txt
   ..\data

---
## Cell 6 — Text Cleaning

In [20]:
# -----------------------------------------------------------------------------
# Text Cleaning and Quality Validation
#   Applied BEFORE chunking to improve embedding quality.
#   The cleaner preserves content; validators skip extraction noise.
# -----------------------------------------------------------------------------

LETTER_TOKEN_RE = re.compile(r"[^\W\d_]{2,}", re.UNICODE)
PAGE_NUMBER_LINE_RE = re.compile(
    r"^(?:page\s*)?\d{1,3}(?:\s*(?:/|sur|of)\s*\d{1,3})?$",
    re.IGNORECASE,
)


def is_page_number_line(line: str) -> bool:
    """Return True for standalone page-number/header artifacts."""
    return bool(PAGE_NUMBER_LINE_RE.fullmatch((line or "").strip()))


def preview_text(text: str, limit: int = 90) -> str:
    """Compact preview for logs."""
    return re.sub(r"\s+", " ", text or "").strip()[:limit]


def clean_text(text: str) -> str:
    """
    Normalize raw extracted text for embedding without stripping useful content.

    Key choices:
      1. Unicode normalization and common invisible-character cleanup.
      2. Remove control/private-use glyph artifacts produced by bad PDF fonts.
      3. Repair line-break hyphenation, but keep real punctuation and accents.
      4. Drop standalone page-number lines only.
      5. Preserve paragraph breaks for recursive chunking.
    """
    if not text:
        return ""

    text = unicodedata.normalize("NFKC", text)
    text = (
        text.replace("\u00a0", " ")
        .replace("\u200b", "")
        .replace("\ufeff", "")
        .replace("\u00ad", "")
    )

    cleaned_chars = []
    for ch in text:
        category = unicodedata.category(ch)
        if ch in "\n\t":
            cleaned_chars.append(ch)
        elif category in {"Cc", "Co", "Cs"}:
            cleaned_chars.append(" ")
        else:
            cleaned_chars.append(ch)
    text = "".join(cleaned_chars)

    text = re.sub(r"(?<=[^\W\d_])-\s*\n\s*(?=[^\W\d_])", "", text, flags=re.UNICODE)
    text = re.sub(r"[ \t]*\n[ \t]*", "\n", text)

    lines: list[str] = []
    for line in text.splitlines():
        line = re.sub(r"[ \t]{2,}", " ", line.strip())
        if not line:
            if lines and lines[-1] != "":
                lines.append("")
            continue
        if is_page_number_line(line):
            continue
        lines.append(line)

    text = "\n".join(lines)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def text_stats(text: str) -> dict:
    """Compute lightweight quality stats for a text block."""
    value = text or ""
    non_space = sum(1 for ch in value if not ch.isspace())
    letters = sum(1 for ch in value if ch.isalpha())
    digits = sum(1 for ch in value if ch.isdigit())
    private_use = sum(1 for ch in value if 0xE000 <= ord(ch) <= 0xF8FF)
    words = LETTER_TOKEN_RE.findall(value)

    return {
        "chars": len(value.strip()),
        "non_space": non_space,
        "letters": letters,
        "digits": digits,
        "words": len(words),
        "letter_ratio": letters / max(non_space, 1),
        "numeric_ratio": digits / max(non_space, 1),
        "private_use_ratio": private_use / max(non_space, 1),
    }


def quality_reason(text: str, *, min_chars: int, min_words: int) -> Optional[str]:
    """Return None if text is informative enough; otherwise return skip reason."""
    value = (text or "").strip()
    if not value:
        return "empty"

    stats = text_stats(value)
    if stats["private_use_ratio"] > CONFIG["max_private_use_ratio"]:
        return "garbled_private_use_glyphs"
    if stats["chars"] < min_chars:
        return f"too_short<{min_chars}"
    if stats["letters"] < max(12, min_chars // 5):
        return "too_few_letters"
    if stats["words"] < min_words:
        return f"too_few_words<{min_words}"
    if stats["numeric_ratio"] > CONFIG["max_numeric_ratio"]:
        return "mostly_numbers"
    if stats["letter_ratio"] < 0.20:
        return "low_letter_ratio"

    compact = re.sub(r"\s+", "", value)
    if len(compact) >= min_chars and len(set(compact)) <= 2:
        return "repeated_characters"

    return None


def is_informative_text(text: str, *, min_chars: int, min_words: int) -> bool:
    """Boolean wrapper around quality_reason()."""
    return quality_reason(text, min_chars=min_chars, min_words=min_words) is None


def filter_informative_pages(pages: list[dict], filename: str) -> list[dict]:
    """Skip pages that are empty, too short, numeric-only, or extraction noise."""
    kept = []
    for page in pages:
        reason = quality_reason(
            page.get("text", ""),
            min_chars=CONFIG["min_page_chars"],
            min_words=CONFIG["min_page_words"],
        )
        if reason:
            logger.info(
                "SKIP page: file=%s page=%s reason=%s chars=%d preview=%r",
                filename,
                page.get("page", -1),
                reason,
                len((page.get("text") or "").strip()),
                preview_text(page.get("text", "")),
            )
            continue
        kept.append(page)
    return kept


def _normalized_noise_line(line: str) -> str:
    """Normalize a candidate header/footer line for comparison."""
    return re.sub(r"\s+", " ", line.strip()).casefold()


def remove_repeated_header_footer(pages: list[dict], min_repeat: Optional[int] = None) -> list[dict]:
    """
    Remove repeated header/footer lines conservatively.

    Only lines in the first/last N lines of each page can be removed. This avoids
    deleting repeated but meaningful body text.
    """
    if len(pages) < 3:
        return pages

    window = CONFIG["header_footer_window"]
    min_repeat = min_repeat or max(3, math.ceil(len(pages) * CONFIG["header_footer_min_page_ratio"]))

    line_counts: Counter = Counter()
    for page in pages:
        lines = [line.strip() for line in page.get("text", "").splitlines() if line.strip()]
        edge_lines = lines[:window] + lines[-window:]
        candidates = set()
        for line in edge_lines:
            normalized = _normalized_noise_line(line)
            if not normalized or is_page_number_line(normalized):
                continue
            if 3 <= len(normalized) <= 120:
                candidates.add(normalized)
        line_counts.update(candidates)

    noise_lines = {line for line, count in line_counts.items() if count >= min_repeat}
    if not noise_lines:
        return pages

    logger.info("Removing %d repeated header/footer line(s)", len(noise_lines))
    cleaned = []
    for page in pages:
        filtered_lines = []
        for line in page.get("text", "").splitlines():
            normalized = _normalized_noise_line(line)
            if normalized in noise_lines or is_page_number_line(line):
                continue
            filtered_lines.append(line)
        updated = dict(page)
        updated["text"] = "\n".join(filtered_lines)
        cleaned.append(updated)
    return cleaned


sample_dirty = "  Hello\u00a0World!\n\n\n   \t Extra spaces  \n1\n"
print("Before:", repr(sample_dirty))
print("After :", repr(clean_text(sample_dirty)))
print("\nText cleaner and quality gates ready.")

Before: '  Hello\xa0World!\n\n\n   \t Extra spaces  \n1\n'
After : 'Hello World!\n\nExtra spaces'

Text cleaner and quality gates ready.


---
## Cell 7 — Chunking with Deduplication

In [21]:
# -----------------------------------------------------------------------------
# Chunking
#   - RecursiveCharacterTextSplitter respects paragraph / sentence boundaries
#   - Quality gates reject tiny/meaningless chunks before embedding
#   - Canonical chunk hashes prevent duplicate vectors
# -----------------------------------------------------------------------------

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CONFIG["chunk_size"],
    chunk_overlap=CONFIG["chunk_overlap"],
    length_function=len,
    separators=["\n\n", "\n", ". ", "; ", ", ", " ", ""],
)


def build_chunks(
    pages: list[dict],
    filepath: str,
    file_hash: str,
    file_type: str,
    version: int,
    existing_chunk_hashes: set[str],
) -> list[dict]:
    """
    Split page texts into validated chunks and attach rich metadata.

    Chunks are skipped when they are empty, too short, numeric-only, garbled,
    already in ChromaDB, or repeated within the current file.
    """
    path_obj = Path(filepath)
    filename = path_obj.name
    source_path = str(path_obj.resolve())
    ingestion_ts = datetime.now(timezone.utc).isoformat()
    chunks_out = []
    chunk_index = 0
    seen_hashes_this_run: set[str] = set()
    skip_counts: Counter = Counter()

    for page_data in pages:
        page_num = page_data.get("page")
        raw_text = (page_data.get("text") or "").strip()
        if not raw_text:
            continue

        split_texts = splitter.split_text(raw_text)

        for text in split_texts:
            text = clean_text(text)
            reason = quality_reason(
                text,
                min_chars=CONFIG["min_chunk_chars"],
                min_words=CONFIG["min_chunk_words"],
            )
            if reason:
                skip_counts[reason] += 1
                logger.info(
                    "SKIP chunk: file=%s page=%s reason=%s chars=%d preview=%r",
                    filename,
                    page_num if page_num is not None else -1,
                    reason,
                    len(text),
                    preview_text(text),
                )
                continue

            chunk_hash = sha256_string(text, canonicalize=True)
            legacy_chunk_hash = sha256_string(text, canonicalize=False)

            if chunk_hash in existing_chunk_hashes or legacy_chunk_hash in existing_chunk_hashes:
                skip_counts["duplicate_existing"] += 1
                logger.info(
                    "SKIP chunk: file=%s page=%s reason=duplicate_existing preview=%r",
                    filename,
                    page_num if page_num is not None else -1,
                    preview_text(text),
                )
                continue

            if chunk_hash in seen_hashes_this_run:
                skip_counts["duplicate_in_file"] += 1
                logger.info(
                    "SKIP chunk: file=%s page=%s reason=duplicate_in_file preview=%r",
                    filename,
                    page_num if page_num is not None else -1,
                    preview_text(text),
                )
                continue

            seen_hashes_this_run.add(chunk_hash)
            chunk_id = f"{file_hash[:12]}_{chunk_index:05d}"
            stats = text_stats(text)

            metadata = {
                "filename": filename,
                "filepath": source_path,
                "file_type": file_type,
                "file_hash": file_hash,
                "chunk_id": chunk_id,
                "chunk_hash": chunk_hash,
                "chunk_hash_algo": "sha256:nfkc_ws_casefold",
                "chunk_index": chunk_index,
                "page": page_num if page_num is not None else -1,
                "extraction_method": page_data.get("extraction_method", "unknown"),
                "text_chars": stats["chars"],
                "text_words": stats["words"],
                "ingestion_ts": ingestion_ts,
                "version": version,
            }

            chunks_out.append({"id": chunk_id, "text": text, "metadata": metadata})
            chunk_index += 1

    if skip_counts:
        logger.info("Chunk skip summary for %s: %s", filename, dict(skip_counts))

    return chunks_out


print(
    "RecursiveCharacterTextSplitter configured "
    f"(chunk_size={CONFIG['chunk_size']}, overlap={CONFIG['chunk_overlap']}, "
    f"min_chunk_chars={CONFIG['min_chunk_chars']})"
)

RecursiveCharacterTextSplitter configured (chunk_size=800, overlap=100, min_chunk_chars=120)


In [ ]:
# build_chunks() is called by ingest_file().
# This placeholder prevents an accidental run-all failure from a scratch call.

---
## Cell 8 — Embedding Model

In [22]:
# -----------------------------------------------------------------------------
# Embedding Model: all-MiniLM-L6-v2
# -----------------------------------------------------------------------------

print(f"Loading embedding model '{CONFIG['embedding_model']}' ...")
embedding_model = SentenceTransformer(CONFIG["embedding_model"])
print(f"Model loaded. Output dimension: {embedding_model.get_sentence_embedding_dimension()}")


def embed_texts(texts: list[str], batch_size: Optional[int] = None) -> list[list[float]]:
    """Encode strings into embedding vectors as plain Python lists."""
    if not texts:
        return []
    if any(not (text or "").strip() for text in texts):
        raise ValueError("embed_texts received an empty string; filter chunks before embedding.")

    vectors = embedding_model.encode(
        texts,
        batch_size=batch_size or CONFIG["embedding_batch_size"],
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    return vectors.tolist()


test_vec = embed_texts(["This is a test sentence."])
print(f"   Sample embedding - first 5 dims: {[round(v, 4) for v in test_vec[0][:5]]}")

Loading embedding model 'all-MiniLM-L6-v2' ...


Loading weights: 100%|███████████████████████████████████████████| 103/103 [00:00<00:00, 2680.42it/s]
D:\temp\ipykernel_24432\791678123.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded. Output dimension: {embedding_model.get_sentence_embedding_dimension()}")


Model loaded. Output dimension: 384
   Sample embedding - first 5 dims: [0.0843, 0.058, 0.0045, 0.1058, 0.0071]


---
## Cell 9 — ChromaDB Connection & Collection Setup

In [23]:
# -----------------------------------------------------------------------------
# ChromaDB - HTTP Client (connects to the Docker container)
# -----------------------------------------------------------------------------

print(f"Connecting to ChromaDB at {CONFIG['chroma_host']}:{CONFIG['chroma_port']} ...")

chroma_client = chromadb.HttpClient(
    host=CONFIG["chroma_host"],
    port=CONFIG["chroma_port"],
)

try:
    chroma_client.heartbeat()
    print("ChromaDB heartbeat OK.")
except Exception as e:
    raise ConnectionError(
        f"Cannot reach ChromaDB at {CONFIG['chroma_host']}:{CONFIG['chroma_port']}. "
        f"Is the Docker container running?\nOriginal error: {e}"
    )

collection = chroma_client.get_or_create_collection(
    name=CONFIG["collection_name"],
    metadata={"hnsw:space": "cosine"},
)

print(f"Collection '{CONFIG['collection_name']}' ready. Current doc count: {collection.count()}")


def get_existing_chunk_hashes() -> set[str]:
    """Fetch all chunk_hash metadata values from ChromaDB."""
    total = collection.count()
    if total == 0:
        return set()

    hashes: set[str] = set()
    offset = 0
    batch = CONFIG["existing_hash_batch_size"]
    while offset < total:
        results = collection.get(limit=batch, offset=offset, include=["metadatas"])
        for meta in results.get("metadatas", []) or []:
            if meta and meta.get("chunk_hash"):
                hashes.add(meta["chunk_hash"])
        offset += batch

    return hashes


def collection_existing_ids(ids: list[str], batch_size: int = 5000) -> set[str]:
    """Return the subset of ids that currently exists in ChromaDB."""
    found: set[str] = set()
    ids = [id_ for id_ in ids if id_]
    for i in range(0, len(ids), batch_size):
        batch = ids[i : i + batch_size]
        if not batch:
            continue
        result = collection.get(ids=batch, include=["metadatas"])
        found.update(result.get("ids", []) or [])
    return found


def registry_entry_is_live(file_hash: str, registry: dict) -> bool:
    """Trust registry hits only when all registered chunk IDs still exist."""
    entry = registry.get(file_hash)
    if not entry:
        return False
    chunk_ids = entry.get("chunk_ids", [])
    if not chunk_ids:
        return False
    return len(collection_existing_ids(chunk_ids)) == len(chunk_ids)


def delete_chunk_ids(chunk_ids: list[str]) -> int:
    """Delete chunk IDs from ChromaDB if they exist; return requested count."""
    chunk_ids = [id_ for id_ in chunk_ids if id_]
    if not chunk_ids:
        return 0
    collection.delete(ids=chunk_ids)
    return len(chunk_ids)


EXISTING_CHUNK_HASHES = get_existing_chunk_hashes()
print(f"   Existing chunk hashes loaded: {len(EXISTING_CHUNK_HASHES)}")

Connecting to ChromaDB at localhost:8000 ...
ChromaDB heartbeat OK.
Collection 'rag_collection_app' ready. Current doc count: 0
   Existing chunk hashes loaded: 0


---
## Cell 10 — Near-Duplicate Detection (Optional, Embedding-Based)

In [24]:
# -----------------------------------------------------------------------------
# Near-Duplicate Detection
# -----------------------------------------------------------------------------

def document_probe_text(pages: list[dict], max_chars: int = CONFIG["near_dup_probe_chars"]) -> str:
    """Build a meaningful probe from the first informative cleaned pages."""
    parts = []
    total = 0
    for page in pages:
        text = (page.get("text") or "").strip()
        if quality_reason(text, min_chars=CONFIG["min_page_chars"], min_words=CONFIG["min_page_words"]):
            continue
        remaining = max_chars - total
        if remaining <= 0:
            break
        parts.append(text[:remaining])
        total += len(parts[-1])
    return "\n\n".join(parts).strip()


def is_near_duplicate(probe_text: str, threshold: float = CONFIG["near_dup_threshold"]) -> tuple[bool, Optional[str]]:
    """Return (is_near_duplicate, matching_filename_or_None)."""
    if collection.count() == 0:
        return False, None

    probe_text = clean_text(probe_text)[: CONFIG["near_dup_probe_chars"]].strip()
    if quality_reason(probe_text, min_chars=CONFIG["min_page_chars"], min_words=CONFIG["min_page_words"]):
        return False, None

    vec = embed_texts([probe_text])
    results = collection.query(query_embeddings=vec, n_results=1, include=["metadatas", "distances"])
    distances = results.get("distances") or []
    if not distances or not distances[0]:
        return False, None

    similarity = 1.0 - distances[0][0]
    meta = (results.get("metadatas") or [[{}]])[0][0] or {}
    if similarity >= threshold:
        return True, meta.get("filename", "unknown")

    return False, None


print(f"Near-duplicate detector ready (cosine threshold = {CONFIG['near_dup_threshold']}).")

Near-duplicate detector ready (cosine threshold = 0.97).


---
## Cell 11 — Main Ingestion Pipeline

This cell ties together all previous cells into a single, robust ingestion loop.
Running it multiple times is **safe** — files already in the registry are skipped.

In [25]:
# -----------------------------------------------------------------------------
# Ingestion Pipeline
# -----------------------------------------------------------------------------

def ingest_file(filepath: str) -> dict:
    """Full ingestion pipeline for a single file."""
    global REGISTRY, EXISTING_CHUNK_HASHES

    filepath = str(filepath)
    path_obj = Path(filepath)
    filename = path_obj.name
    file_type = path_obj.suffix.lower().lstrip(".")

    try:
        file_hash = sha256_file(filepath)
    except Exception as e:
        logger.error("Cannot hash '%s': %s", filename, e)
        return {"status": "error", "chunks_added": 0, "near_duplicate": False}

    if is_file_already_ingested(file_hash, REGISTRY):
        if registry_entry_is_live(file_hash, REGISTRY):
            print(f"  SKIP exact duplicate: {filename}")
            return {"status": "skipped_exact_dup", "chunks_added": 0, "near_duplicate": False}

        logger.warning(
            "Registry entry for '%s' is stale or ChromaDB is missing chunks; re-ingesting.",
            filename,
        )
        old_chunk_ids = REGISTRY.get(file_hash, {}).get("chunk_ids", [])
        try:
            delete_chunk_ids(old_chunk_ids)
        except Exception as e:
            logger.warning("Could not delete stale chunks for '%s': %s", filename, e)
        REGISTRY.pop(file_hash, None)
        save_registry(REGISTRY, CONFIG["registry_path"])
        EXISTING_CHUNK_HASHES = get_existing_chunk_hashes()

    pages = load_document(filepath)
    if not pages:
        logger.warning("EMPTY/UNREADABLE: %s", filename)
        return {"status": "empty", "chunks_added": 0, "near_duplicate": False}

    if file_type == "pdf":
        pages = remove_repeated_header_footer(pages)

    cleaned_pages = []
    for page in pages:
        updated = dict(page)
        updated["text"] = clean_text(updated.get("text", ""))
        cleaned_pages.append(updated)

    pages = filter_informative_pages(cleaned_pages, filename)
    if not pages:
        logger.warning("ALL PAGES EMPTY/LOW-QUALITY after cleaning: %s", filename)
        return {"status": "empty", "chunks_added": 0, "near_duplicate": False}

    probe_text = document_probe_text(pages)
    is_near_dup, match_name = is_near_duplicate(probe_text)
    if is_near_dup:
        logger.warning(
            "NEAR-DUPLICATE: '%s' is very similar to '%s' (cosine >= %.3f). Proceeding as a new version.",
            filename,
            match_name,
            CONFIG["near_dup_threshold"],
        )

    version = get_file_version(REGISTRY, filename)
    chunks = build_chunks(
        pages=pages,
        filepath=filepath,
        file_hash=file_hash,
        file_type=file_type,
        version=version,
        existing_chunk_hashes=EXISTING_CHUNK_HASHES,
    )

    if not chunks:
        logger.warning("No valid new chunks for '%s' after filtering/deduplication.", filename)
        return {"status": "skipped_no_valid_chunks", "chunks_added": 0, "near_duplicate": is_near_dup}

    batch_size = CONFIG["chroma_batch_size"]
    for i in range(0, len(chunks), batch_size):
        batch = chunks[i : i + batch_size]
        batch_texts = [c["text"] for c in batch]
        batch_embeddings = embed_texts(batch_texts, batch_size=CONFIG["embedding_batch_size"])
        collection.add(
            ids=[c["id"] for c in batch],
            documents=batch_texts,
            embeddings=batch_embeddings,
            metadatas=[c["metadata"] for c in batch],
        )

    for c in chunks:
        EXISTING_CHUNK_HASHES.add(c["metadata"]["chunk_hash"])

    REGISTRY[file_hash] = {
        "filename": filename,
        "filepath": str(path_obj.resolve()),
        "file_type": file_type,
        "version": version,
        "ingested_at": datetime.now(timezone.utc).isoformat(),
        "page_count": len(pages),
        "chunk_count": len(chunks),
        "chunk_ids": [c["id"] for c in chunks],
        "near_duplicate": is_near_dup,
        "near_duplicate_match": match_name,
    }
    save_registry(REGISTRY, CONFIG["registry_path"])

    print(f"  INGESTED: {filename} [v{version} | {len(chunks)} chunks | hash: {file_hash[:12]}...]")
    return {"status": "ingested", "chunks_added": len(chunks), "near_duplicate": is_near_dup}


print(f"\n{'=' * 60}")
print(f"  Starting ingestion - {len(all_files)} file(s) found")
print(f"{'=' * 60}\n")

summary = {
    "ingested": 0,
    "skipped_exact_dup": 0,
    "skipped_no_valid_chunks": 0,
    "empty": 0,
    "error": 0,
    "near_duplicate_warnings": 0,
}

for fpath in tqdm(all_files, desc="Ingesting", unit="file"):
    result = ingest_file(fpath)
    status = result.get("status", "error")
    summary[status] = summary.get(status, 0) + 1
    if result.get("near_duplicate"):
        summary["near_duplicate_warnings"] += 1

print(f"\n{'=' * 60}")
print("  Ingestion complete - Summary")
print(f"{'=' * 60}")
for key, val in summary.items():
    print(f"   {key}: {val}")
print(f"   Total docs in collection: {collection.count()}")


  Starting ingestion - 25 file(s) found



Ingesting:   0%|                                                            | 0/25 [00:00<?, ?file/s]WARNING | Registry entry for '130042010150605.pdf' is stale or ChromaDB is missing chunks; re-ingesting.
INFO | SKIP page: file=130042010150605.pdf page=1 reason=empty chars=0 preview=''
INFO | SKIP chunk: file=130042010150605.pdf page=14 reason=too_short<120 chars=108 preview='viande étrangères ; • Etude de l’effet de l’alimentation sur la qualité nutritionnelle de '
INFO | Chunk skip summary for 130042010150605.pdf: {'too_short<120': 1}
Ingesting:   4%|██                                                  | 1/25 [00:07<02:56,  7.34s/file]WARNING | Registry entry for '130828-veille_huile_dolive-sl.pdf' is stale or ChromaDB is missing chunks; re-ingesting.


  INGESTED: 130042010150605.pdf [v1 | 54 chunks | hash: e338598ee95e...]


INFO | Removing 1 repeated header/footer line(s)
Ingesting:   8%|████▏                                               | 2/25 [00:13<02:30,  6.53s/file]WARNING | Registry entry for '19-00145-book_agricultures_en_chiffres_def.pdf' is stale or ChromaDB is missing chunks; re-ingesting.


  INGESTED: 130828-veille_huile_dolive-sl.pdf [v1 | 65 chunks | hash: 818c7321b47e...]


...

INFO | SKIP chunk: file=fiche_region_fes_meknes.pdf page=7 reason=too_short<120 chars=112 preview='29,32 Apiculture 27,42 Lait 14,31 Olivier 10,93 Arboriculture fruitière 5,28 PAM 3,33 Avic'
INFO | Chunk skip summary for fiche_region_fes_meknes.pdf: {'too_short<120': 1}
Ingesting:  96%|████████████████████████████████████████████████▉  | 24/25 [04:50<00:13, 13.40s/file]WARNING | Registry entry for 'produits_terroir_edition2014_fr.pdf' is stale or ChromaDB is missing chunks; re-ingesting.


  INGESTED: fiche_region_fes_meknes.pdf [v1 | 18 chunks | hash: 68c1f49e2243...]


INFO | SKIP chunk: file=produits_terroir_edition2014_fr.pdf page=13 reason=too_short<120 chars=109 preview='aux produits animaux lait et viandes des goûts distingués et en étroite relation avec les '
INFO | SKIP chunk: file=produits_terroir_edition2014_fr.pdf page=37 reason=too_short<120 chars=112 preview='l’occurrence la fabrication de la confiture, le jus, l’huile extraite des grains de fruits'
INFO | SKIP chunk: file=produits_terroir_edition2014_fr.pdf page=38 reason=too_short<120 chars=69 preview='Lait de chamelle Miel d’euphorbe Dénomination Appellation en Français'
INFO | SKIP chunk: file=produits_terroir_edition2014_fr.pdf page=38 reason=too_short<120 chars=102 preview='Usages Le miel d’euphorbe est consommé frais. Il possède des propriétés médicinales et thé'
INFO | SKIP chunk: file=produits_terroir_edition2014_fr.pdf page=40 reason=too_short<120 chars=59 preview='Palmier dattier Agneau Siroua Dénomination Nom Scientifique'
INFO | SKIP chunk: file=produits_terroir_edition2014_f

  INGESTED: produits_terroir_edition2014_fr.pdf [v1 | 328 chunks | hash: e02454df3a7b...]



  Ingestion complete - Summary
   ingested: 23
   skipped_exact_dup: 0
   skipped_no_valid_chunks: 0
   empty: 2
   error: 0
   near_duplicate_warnings: 0
   Total docs in collection: 2991


---
## Cell 12 — Retrieval Test

In [26]:
# -----------------------------------------------------------------------------
# Retrieval Test
# -----------------------------------------------------------------------------

def retrieve(query: str, top_k: int = CONFIG["top_k"], where: Optional[dict] = None) -> list[dict]:
    """Semantic similarity search against ChromaDB."""
    query = (query or "").strip()
    if not query:
        raise ValueError("retrieve() requires a non-empty query")

    total = collection.count()
    if total == 0 or top_k <= 0:
        return []

    kwargs = dict(
        query_embeddings=embed_texts([query]),
        n_results=min(top_k, total),
        include=["documents", "metadatas", "distances"],
    )
    if where:
        kwargs["where"] = where

    raw = collection.query(**kwargs)
    docs = (raw.get("documents") or [[]])[0]
    metas = (raw.get("metadatas") or [[]])[0]
    distances = (raw.get("distances") or [[]])[0]

    results = []
    for doc, meta, dist in zip(docs, metas, distances):
        results.append({"text": doc, "metadata": meta or {}, "similarity_score": round(1.0 - dist, 4)})
    return results


def print_results(results: list[dict]) -> None:
    """Pretty-print retrieval results."""
    if not results:
        print("No retrieval results found.")
        return

    for i, r in enumerate(results, 1):
        m = r["metadata"]
        print(f"\n{'-' * 60}")
        print(f"  Result #{i}   score={r['similarity_score']}")
        print(f"  File   : {m.get('filename')}  (v{m.get('version')})")
        print(f"  Type   : {m.get('file_type')}   Page: {m.get('page')}")
        print(f"  Hash   : {m.get('file_hash', '')[:16]}...")
        print(f"  Ingested: {m.get('ingestion_ts')}")
        print(f"  Preview: {preview_text(r['text'], 300)} ...")
    print(f"\n{'-' * 60}")


SAMPLE_QUERY = "ariculture dans le maroc?"
print(f"Query: \"{SAMPLE_QUERY}\"")
print(f"Top-{CONFIG['top_k']} results:\n")

if collection.count() == 0:
    print("Collection is empty. Ingest some files first (Cell 11).")
else:
    results = retrieve(SAMPLE_QUERY)
    print_results(results)

Query: "ariculture dans le maroc?"
Top-5 results:


------------------------------------------------------------
  Result #1   score=0.6354
  File   : agriculture_maroc_fruits_synthetic-data.txt  (v1)
  Type   : txt   Page: -1
  Hash   : 3f587126a05ddaf3...
  Ingested: 2026-05-05T18:36:03.889856+00:00
  Preview: Général | Tout Maroc | Conseil d'expert : un quart des pertes post-récolte au Maroc sont dues à des ruptures de la chaîne du froid, un chantier prioritaire ...

------------------------------------------------------------
  Result #2   score=0.6329
  File   : Filière Truffes au Maroc VF btt_213.pdf  (v1)
  Type   : pdf   Page: 15
  Hash   : e7f0cc0bba34aa31...
  Ingested: 2026-05-05T18:35:12.579648+00:00
  Preview: truffe marocaine a conquis les marchés du Moyen Orient et d’Europe a conduit à une augmentation remarquable des prix de vente. Certains de ces pays importateurs sont aussi producteurs des mêmes espèces que le Maroc mais leurs productions ont diminué, pour les uns à c

In [ ]:
# -----------------------------------------------------------------------------
# Optional Admin Utility: Reset the ChromaDB collection and registry
# -----------------------------------------------------------------------------

def reset_collection(confirm: bool = False) -> None:
    """
    Delete the configured ChromaDB collection and clear the local registry.
    This is intentionally opt-in so running the notebook cannot wipe data by accident.
    """
    global collection, REGISTRY, EXISTING_CHUNK_HASHES

    if not confirm:
        print("Reset skipped. Call reset_collection(confirm=True) to delete the collection and registry.")
        return

    try:
        chroma_client.delete_collection(name=CONFIG["collection_name"])
    except Exception as e:
        logger.warning("Collection delete raised an error, continuing with recreate: %s", e)

    collection = chroma_client.get_or_create_collection(
        name=CONFIG["collection_name"],
        metadata={"hnsw:space": "cosine"},
    )
    REGISTRY = {}
    EXISTING_CHUNK_HASHES = set()
    save_registry(REGISTRY, CONFIG["registry_path"])
    print(f"Collection '{CONFIG['collection_name']}' and registry reset.")


print("Optional reset utility ready. Nothing is deleted unless reset_collection(confirm=True) is called.")

---
## Cell 13 — RAG Function: Retrieval → Prompt Builder

This function is the final building block of a RAG system.
It retrieves relevant chunks and formats a **context-enriched prompt**
ready to be sent to any LLM (OpenAI, Anthropic, Ollama, etc.).

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# RAG Function
# ─────────────────────────────────────────────────────────────────────────────

RAG_SYSTEM_PROMPT = """\
You are a knowledgeable assistant. Answer the user's question using ONLY the \
context provided below. If the context does not contain enough information, \
say so clearly instead of guessing.
"""


def build_rag_prompt(
    query: str,
    top_k: int = CONFIG["top_k"],
    where: Optional[dict] = None,
    max_context_chars: int = 4000,
) -> dict:
    """
    End-to-end RAG function.

    1. Retrieves top-k relevant chunks from ChromaDB.
    2. Formats retrieved chunks into a numbered context block.
    3. Returns a dict with:
       - system_prompt : str  — instruct the LLM how to behave
       - user_prompt   : str  — context + question ready for the LLM
       - sources       : list — metadata for citations / traceability
       - retrieved_chunks: list — raw results for debugging

    Drop the returned dict straight into any LLM SDK::

        # OpenAI example
        rag = build_rag_prompt("Explain X")
        response = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system",  "content": rag["system_prompt"]},
                {"role": "user",    "content": rag["user_prompt"]},
            ],
        )

        # Anthropic example
        response = anthropic_client.messages.create(
            model="claude-sonnet-4-20250514",
            system=rag["system_prompt"],
            messages=[{"role": "user", "content": rag["user_prompt"]}],
            max_tokens=1024,
        )
    """
    if collection.count() == 0:
        return {
            "system_prompt": RAG_SYSTEM_PROMPT,
            "user_prompt": f"Question: {query}\n\nContext: [No documents ingested yet.]",
            "sources": [],
            "retrieved_chunks": [],
        }

    # ── 1. Retrieve ───────────────────────────────────────────────────────
    chunks = retrieve(query, top_k=top_k, where=where)

    # ── 2. Format context block (respect max_context_chars budget) ─────────
    context_parts = []
    sources = []
    char_budget = max_context_chars

    for idx, chunk in enumerate(chunks, 1):
        m = chunk["metadata"]
        header = (
            f"[{idx}] Source: {m.get('filename')} "
            f"(page {m.get('page')}, v{m.get('version')}, "
            f"score={chunk['similarity_score']})"
        )
        body = chunk["text"]

        entry = f"{header}\n{body}"

        if len(entry) > char_budget:
            # Truncate to remaining budget
            entry = entry[:char_budget] + " [...truncated]"
            context_parts.append(entry)
            sources.append(m)
            break

        context_parts.append(entry)
        sources.append(m)
        char_budget -= len(entry)

    context_str = "\n\n".join(context_parts)

    # ── 3. Assemble prompt ────────────────────────────────────────────────
    user_prompt = (
        f"Context (retrieved from document store):\n"
        f"{'─'*50}\n"
        f"{context_str}\n"
        f"{'─'*50}\n\n"
        f"Question: {query}\n\n"
        f"Please answer based solely on the context above."
    )

    return {
        "system_prompt":    RAG_SYSTEM_PROMPT,
        "user_prompt":      user_prompt,
        "sources":          sources,
        "retrieved_chunks": chunks,
    }


# ── Demo ──────────────────────────────────────────────────────────────────────
rag_output = build_rag_prompt(SAMPLE_QUERY)

print("=" * 60)
print("SYSTEM PROMPT")
print("=" * 60)
print(rag_output["system_prompt"])

print("\n" + "=" * 60)
print("USER PROMPT (first 1200 chars)")
print("=" * 60)
print(rag_output["user_prompt"][:1200])

print("\n" + "=" * 60)
print(f"SOURCES ({len(rag_output['sources'])} chunk(s))")
print("=" * 60)
for s in rag_output["sources"]:
    print(f"  • {s.get('filename')}  page={s.get('page')}  v{s.get('version')}")

print("\n✅ RAG prompt ready. Pass system_prompt + user_prompt to your LLM of choice.")

---
## Cell 14 — Registry Inspection & Pipeline Stats

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Inspect the ingestion registry and collection stats
# Useful for debugging, auditing, or monitoring.
# ─────────────────────────────────────────────────────────────────────────────

print("=" * 60)
print("  INGESTION REGISTRY")
print("=" * 60)

current_registry = load_registry(CONFIG["registry_path"])

if not current_registry:
    print("  Registry is empty. Run Cell 11 to ingest files.")
else:
    for file_hash, info in current_registry.items():
        print(
            f"  [{file_hash[:12]}...]  "
            f"{info['filename']}  "
            f"v{info['version']}  "
            f"{info['chunk_count']} chunks  "
            f"ingested: {info['ingested_at'][:19]}"
        )

print(f"\n  Total tracked files : {len(current_registry)}")
print(f"  Total chunks in DB  : {collection.count()}")


# ── Show version history for files that were re-ingested ─────────────────────
from collections import defaultdict

version_map: dict = defaultdict(list)
for fhash, info in current_registry.items():
    version_map[info["filename"]].append(info["version"])

versioned = {k: v for k, v in version_map.items() if max(v) > 1}
if versioned:
    print("\n  Files with multiple versions:")
    for fname, versions in versioned.items():
        print(f"    {fname}: versions {sorted(versions)}")
else:
    print("\n  No files with multiple versions found.")

---
## Cell 15 — Utility: Delete a File's Chunks from ChromaDB

In [ ]:
# -----------------------------------------------------------------------------
# Utility: Remove a specific file from the vector DB and registry.
# -----------------------------------------------------------------------------

def delete_file_from_db(filename: str) -> int:
    """Remove all chunks for the given filename from ChromaDB and the registry."""
    reg = load_registry(CONFIG["registry_path"])
    matching_hashes = [fhash for fhash, info in reg.items() if info.get("filename") == filename]

    if not matching_hashes:
        print(f"  '{filename}' not found in registry.")
        return 0

    total_deleted = 0
    for fhash in matching_hashes:
        chunk_ids = reg[fhash].get("chunk_ids", [])
        if chunk_ids:
            total_deleted += delete_chunk_ids(chunk_ids)
        del reg[fhash]

    save_registry(reg, CONFIG["registry_path"])

    global REGISTRY, EXISTING_CHUNK_HASHES
    REGISTRY = reg
    EXISTING_CHUNK_HASHES = get_existing_chunk_hashes()

    print(f"  Deleted {total_deleted} chunk(s) for '{filename}'.")
    return total_deleted


# Example usage (uncomment to run):
# delete_file_from_db("my_old_document.pdf")

print("delete_file_from_db() utility ready.")

---
## Architecture Summary

```
┌─────────────────────────────────────────────────────────────┐
│                    RAG Ingestion Pipeline                    │
│                                                             │
│  ./data/                                                    │
│  ├── *.pdf   ──► load_pdf()  ──► pages[]                   │
│  ├── *.txt   ──► load_txt()  ──►  │                        │
│  └── *.docx  ──► load_docx() ──►  │                        │
│                                   ▼                        │
│                           clean_text()                      │
│                           remove_repeated_header_footer()   │
│                                   │                        │
│            SHA256 file hash ──────┤                        │
│            Registry check ────────┤ ── skip if seen        │
│            Near-dup check ─────────┤ ── warn if similar    │
│                                   │                        │
│                           RecursiveCharacterTextSplitter    │
│                           chunk_size=800 / overlap=100      │
│                                   │                        │
│                     SHA256 chunk hash ── skip if seen      │
│                                   │                        │
│                   SentenceTransformer (all-MiniLM-L6-v2)   │
│                   384-dim L2-normalised embeddings          │
│                                   │                        │
│                          ChromaDB (Docker :8000)            │
│                          collection: rag_collection         │
│                          cosine similarity index            │
│                                   │                        │
│                     retrieve() ─► build_rag_prompt()        │
│                                   │                        │
│                              LLM of choice                  │
└─────────────────────────────────────────────────────────────┘
```

### Data Stock Problem Handling

| Problem | Solution |
|---|---|
| Exact duplicate file | SHA-256 file hash → registry check → skip |
| Near-duplicate file | Embedding cosine similarity query → warn + version |
| Re-ingestion of same dataset | Registry JSON persists across restarts |
| Duplicate chunks across files | SHA-256 chunk hash → skip before insertion |
| File versioning | `get_file_version()` auto-increments per filename |
| Corrupt/empty files | Try/except in loaders → graceful skip |